# I-LOFAR (REALTA) dynamic spectrum

This notebook plots solar dynamic spectra recorded by the **REALTA** backend
of the Irish LOFAR station (I-LOFAR), stored as `sigproc` filterbank
(`.fil`) files. REALTA observes the Sun in three LOFAR observing modes
simultaneously:

- **Mode 3** (LBA Outer):  10 - 90 MHz, 199 channels
- **Mode 5** (HBA Low):  110 - 190 MHz, 200 channels
- **Mode 7** (HBA Mid):  210 - 270 MHz,  88 channels

The three modes are written into a single `.fil` file with frequency gaps
between bands; the helper `freq_axis` below inserts NaNs in those gaps so the
three sub-bands plot correctly on a single axis.

A Stokes parameter (`I` or `V`) can be selected via the `stokes` variable;
the corresponding file is loaded.

**Workflow:** locate the `.fil` file &rarr; read header and data with
`sigpyproc` &rarr; build per-mode (time, freq) DataFrames &rarr; downsample to
1 s &rarr; subtract per-channel median background &rarr; plot the three modes
together on a single log-frequency axis.


In [ ]:
import warnings
warnings.filterwarnings('ignore')

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
from matplotlib.ticker import AutoMinorLocator
from matplotlib.colors import LogNorm
import matplotlib as mpl

# Use the precise matplotlib epoch (avoids ~10 us offsets in old matplotlib).
mpl.rcParams['date.epoch'] = '1970-01-01T00:00:00'
try:
    mdates.set_epoch('1970-01-01T00:00:00')
except RuntimeError:
    pass

# Unified plotting style for all dynamic spectra notebooks.
plt.rcParams['figure.dpi'] = 100
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['xtick.labelsize'] = 11
plt.rcParams['ytick.labelsize'] = 11

from sigpyproc.readers import FilReader
from astropy.time import Time
import astropy.units as u
from datetime import datetime
from tqdm import tqdm


## Configuration


In [ ]:
# Choose 'I' for total intensity or 'V' for circular polarisation.
stokes = 'I'

# Observation date (used to filter files in the data directory).
mydate = '2024-05-14'

data_dir = './sample_data/ilofar'
outputs  = './outputs'
os.makedirs(outputs, exist_ok=True)


## Helper functions


In [ ]:
def subtract_background_median(df):
    """
    Subtract a per-channel median background from a dynamic spectrum.

    The function computes the median along the time axis (axis=0) for each
    frequency channel, then subtracts it from every time sample of that
    channel. This is the standard approach for highlighting transient
    emission against a slowly-varying instrumental/sky background.

    Parameters
    ----------
    df : pandas.DataFrame
        DataFrame with shape (n_times, n_freqs). Index is time, columns
        are frequencies.

    Returns
    -------
    pandas.DataFrame
        Same shape as input with the per-channel median removed.
    """
    bkg = np.nanmedian(df.values, axis=0)
    return df - np.tile(bkg, (df.shape[0], 1))


def freq_axis(freqs):
    """
    Insert the inter-mode frequency gaps for REALTA mode 3+5+7 recordings.

    REALTA writes the three LOFAR observing modes into a single contiguous
    filterbank, but in frequency they are not contiguous (90-110 MHz and
    190-210 MHz gaps). This function returns a new frequency vector with
    those gaps inserted as evenly-spaced filler channels (0.390625 MHz
    wide, the LOFAR channel width), so that pcolormesh shows the gaps
    explicitly.

    Parameters
    ----------
    freqs : numpy.ndarray
        Original 1D frequency vector read from `data.header.chan_freqs`.

    Returns
    -------
    numpy.ndarray
        Frequency vector with gap fillers inserted.
    """
    gap1 = np.flipud(freqs[288] + (np.arange(59) * 0.390625))
    gap2 = np.flipud(freqs[88]  + (np.arange(57) * 0.390625))
    extra = 59 + 57 - 1
    new_freq = np.zeros(extra + freqs.shape[0])

    new_freq[0:88]    = freqs[0:88]
    new_freq[88:145]  = gap2[:57]
    new_freq[145:345] = freqs[88:288]
    new_freq[345:404] = gap1[:59]
    new_freq[404:]    = freqs[289:]

    return new_freq


def parse_realta_filename(path):
    """
    Extract the Stokes label and date/datetime token from a REALTA file path.

    REALTA file names follow patterns such as
        Sun357_<YYYY-MM-DD>T<hhmm>_stokes<I|V>_*.fil

    Returns a (stokes, date_or_datetime) tuple where either element may be
    None if not found.
    """
    fname = os.path.basename(path)
    parts = fname.split('_')
    stokes = None
    date_or_datetime = None
    for p in parts:
        if p.startswith('stokes'):
            stokes = p.replace('stokes', '')
        elif 'T' in p and len(p) == 13 and p[:8].isdigit() and p[9:].isdigit():
            date_or_datetime = p
        elif len(p) == 8 and p.isdigit():
            date_or_datetime = p
    return stokes, date_or_datetime


## Locate the REALTA `.fil` file


In [ ]:
# Filter on date and Stokes; falls back to first match otherwise.
date_compact = mydate.replace('-', '')
candidates = sorted(glob.glob(f'{data_dir}/*.fil'))
matching = [f for f in candidates
            if date_compact in os.path.basename(f).replace('-', '')
            and f'stokes{stokes}' in os.path.basename(f)]
if not matching:
    matching = candidates
print('Found:', *matching, sep='\n  ')

filename = matching[0]
file_stokes, date_token = parse_realta_filename(filename)
print(f'\nUsing file: {filename}')
print(f'Parsed Stokes: {file_stokes}, date token: {date_token}')


## Read header and data


In [ ]:
reader = FilReader(filename)
header = reader.header.to_dict()
display(header)

n_samples = reader.header.nsamples
data = reader.read_block(start=0, nsamps=n_samples)  # shape: (freq, time)
print(f'Data shape (freq, time): {data.shape}')


## Build the time axis


In [ ]:
tstart = Time(data.header.tstart, format='mjd')
tarray = tstart + (np.arange(data.shape[1]) * data.header.tsamp * u.s)

dt_ms = (datetime.strptime(tarray[1].iso,  '%Y-%m-%d %H:%M:%S.%f')
       - datetime.strptime(tarray[0].iso,  '%Y-%m-%d %H:%M:%S.%f')
        ).total_seconds() * 1000
print(f'Sample cadence: {dt_ms:.2f} ms')
print(f'Time range: {tarray[0].iso} -> {tarray[-1].iso}')

# Convert to Python datetimes; cache to disk because the loop is slow.
cache = f'{data_dir}/timesArray_realta_{date_token}.npy'
if os.path.exists(cache):
    Tarray = np.load(cache, allow_pickle=True)
else:
    Tarray = []
    for t in tqdm(tarray, desc='Converting time array'):
        Tarray.append(datetime.strptime(t.iso, '%Y-%m-%d %H:%M:%S.%f'))
    np.save(cache, np.array(Tarray, dtype=object))


## Build the frequency axis and split into modes 3/5/7


In [ ]:
freqs = data.header.chan_freqs
new_freq = freq_axis(freqs)
print(f'Frequency range: {freqs[0]:.2f} - {freqs[-1]:.2f} MHz '
      f'(after inserting gaps: {len(new_freq)} channels)')

# Convert to log10(power) and replace -inf produced by log10(0).
data_log = np.log10(data)
data_log[np.isinf(data_log)] = 0.0

# Insert the gap rows (filled with NaN) so the three sub-bands stay visually
# separate; only the three real-data regions get populated.
data2 = np.full((new_freq.shape[0], data_log.shape[1]), np.nan, dtype=float)
data2[0:88]    = data_log[0:88]
data2[145:345] = data_log[88:288]
data2[404:]    = data_log[289:]

# Canonical (calibrated) frequency vectors for the three modes.
freq_mode3 = np.linspace(10, 90, 199)
freq_mode5 = np.linspace(110, 190, 200)
freq_mode7 = np.linspace(210, 270, 88)

df_mode3 = pd.DataFrame(data=data2[404:].T,    columns=freq_mode3[::-1])
df_mode5 = pd.DataFrame(data=data2[145:345].T, columns=freq_mode5[::-1])
df_mode7 = pd.DataFrame(data=data2[:88].T,     columns=freq_mode7[::-1])

for df in (df_mode3, df_mode5, df_mode7):
    df.index = pd.to_datetime(Tarray)

print(f'Mode 3 shape: {df_mode3.shape}')
print(f'Mode 5 shape: {df_mode5.shape}')
print(f'Mode 7 shape: {df_mode7.shape}')


## Downsample, remove background, and plot


In [ ]:
# Downsample to 1 s for plotting.
df_mode3_1s = df_mode3.resample('1S').mean()
df_mode5_1s = df_mode5.resample('1S').mean()
df_mode7_1s = df_mode7.resample('1S').mean()

# Per-channel median background subtraction.
df_mode3_nobkg = subtract_background_median(df_mode3_1s)
df_mode5_nobkg = subtract_background_median(df_mode5_1s)
df_mode7_nobkg = subtract_background_median(df_mode7_1s)


In [ ]:
# Common colour limits for all three modes so the joint plot is consistent.
vmin = -0.1
vmax = 1.7

fig = plt.figure(figsize=(12, 7))
ax  = fig.add_subplot(111)

for df in (df_mode3_nobkg, df_mode5_nobkg, df_mode7_nobkg):
    pc = ax.pcolormesh(df.index, df.columns, df.values.T,
                       vmin=vmin, vmax=vmax, cmap='Spectral_r')

fig.colorbar(pc, ax=ax, pad=0.02, label='log$_{10}$(power) (background subtracted)')

ax.set_yscale('log')
yticks = [10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 200, 300]
ax.set_yticks(yticks)
ax.set_yticklabels([str(t) for t in yticks])
ax.yaxis.set_minor_locator(AutoMinorLocator(n=10))
ax.set_ylim(bottom=10, top=300)
ax.set_ylim(ax.get_ylim()[::-1])  # descending

ax.set_xlabel('Time (UT)')
ax.set_ylabel('Frequency (MHz)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%H:%M'))
ax.set_title(f'I-LOFAR / REALTA  -  Stokes {file_stokes}  -  {mydate}'
             '   (modes 3 + 5 + 7)')

fig.tight_layout()
fig.savefig(f'{outputs}/ilofar_dyspec_{mydate}_stokes{file_stokes}.png',
            bbox_inches='tight')
plt.show()